# Cyclope — Pre-Fill Pipeline

Notebook consolidado del flujo de onboarding corporativo (Cámara de Comercio de Bogotá).

Tres secciones, ejecutar **en orden de arriba hacia abajo**:

1. **Sección 1 — Extracción del PDF** &nbsp;→&nbsp; lee los PDF de la Cámara y genera `camara_extraction.xlsx` + `.json`.
2. **Sección 2 — Pre-llenado** &nbsp;→&nbsp; lee ese Excel y genera 7 documentos por empresa (DOCX, PDF y XLSX).
3. **Sección 3 — Folder Creation** &nbsp;→&nbsp; crea la carpeta de cada contraparte en la red y copia los documentos.

---

### Librerías de terceros
| Librería | Sección | Instalación |
|---|---|---|
| `pandas`, `openpyxl` | 1, 2 | `pip install pandas openpyxl` |
| `pdfplumber` | 1 | `pip install pdfplumber` |
| `lxml` | 2 | `pip install lxml` |
| `pypdf` | 2 | `pip install pypdf` |

### Uso
1. Edita **solo** la celda de Configuración (rutas).
2. Ejecuta las celdas en orden (`Run All` o una por una).


## ⚙️ 0. Configuración — editar SOLO esta celda al cambiar de equipo

In [ ]:
# ============================================================================
# CONFIGURACIÓN — EDITAR SOLO ESTA CELDA AL CAMBIAR DE EQUIPO
# ============================================================================
# Cambia PROJECT_ROOT (y NETWORK_BASE_PATH si aplica). Todo lo demás se deriva
# automáticamente; no necesitas tocar rutas en ninguna otra celda.
# ============================================================================

import re
import json
import shutil
import zipfile
from pathlib import Path
from datetime import date

import pandas as pd

# ── Carpetas base ────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project")
PDF_INPUT_DIR    = PROJECT_ROOT / "Camara_PDF"     # PDFs de la Cámara (entrada §1)
REQUIREMENTS_DIR = PROJECT_ROOT / "Requirements"   # plantillas de documentos (§2)
OUTPUT_DIR       = PROJECT_ROOT / "Output"         # resultados (Excel, JSON y documentos)

# Carpeta de red donde §3 crea los folders de cada contraparte
NETWORK_BASE_PATH = Path(r"\\naeast.ad.jpmorganchase.com\cib2\dipls\NACIB2DIPLSSHARE00001\COB\Counterparties")

# ── Derivados (no editar) ────────────────────────────────────────────────────
EXCEL_PATH = OUTPUT_DIR / "camara_extraction.xlsx"   # lo genera §1; lo consumen §2 y §3

ANNEX_A_TEMPLATE = REQUIREMENTS_DIR / "Annex A - Signature Card for Accounts in COP - Colombia (English and Spanish).docx"
CONTACT_TEMPLATE = REQUIREMENTS_DIR / "Contact Information Certificate For Electronic Signatures.docx"
CRS_TEMPLATE     = REQUIREMENTS_DIR / "CRS FATCA Form.pdf"
ESST_TEMPLATE    = REQUIREMENTS_DIR / "Electronic Signatures Services Terms.docx"
FORM_GE_TEMPLATE = REQUIREMENTS_DIR / "Formulario Grandes Exposiciones.xlsx"
GCC_TEMPLATE     = REQUIREMENTS_DIR / "GCC-Declaracion de Veracidad.pdf"
KYC_TEMPLATE     = REQUIREMENTS_DIR / "KYC application form (Onboarding) v. Mar 2026.pdf"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuración cargada.")
print(f"  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"  OUTPUT_DIR   : {OUTPUT_DIR}")


# 1️⃣ Sección 1 — Extracción del PDF (Cámara de Comercio)

Lee los PDF de `PDF_INPUT_DIR` y escribe `camara_extraction.xlsx` (hojas **Resumen** y **Nombramientos**) + `camara_extraction.json` en `OUTPUT_DIR`.

In [ ]:
# ============================================================================
# SECCIÓN 1 — Funciones de extracción (de extract_camara.py)
# ============================================================================
# Tercero NUEVO en esta sección:
#   pdfplumber  - Lectura de texto y posiciones de los PDF  -> pip install pdfplumber
# (re, json, pathlib, pandas ya vienen de la celda de Configuración)
# ============================================================================
from __future__ import annotations

import unicodedata
from dataclasses import dataclass, asdict
from typing import Any

import pdfplumber


DOC_ID_RE = re.compile(r"\b(C\.?\s*C\.?|C\.?\s*E\.?|P\.?\s*P\.?)\s*No\.?\s*([A-Z0-9.\-]+)", re.I)

STOP_HEADINGS = {
    "CAMARA DE COMERCIO DE BOGOTA",
    "SEDE VIRTUAL",
    "CERTIFICADO DE EXISTENCIA Y REPRESENTACION LEGAL",
    "CERTIFICADO DE INSCRIPCION DE DOCUMENTOS",
    "FECHA EXPEDICION",
    "RECIBO NO",
    "VALOR",
    "CODIGO DE VERIFICACION",
    "VERIFIQUE EL CONTENIDO",
    "PAGINA",
    "NOMBRAMIENTOS",
    "ORGANO DE ADMINISTRACION",
    "JUNTA DIRECTIVA",
    "REPRESENTANTES LEGALES",
    "REFORMAS DE ESTATUTOS",
    "REVISORES FISCALES",
    "PODERES",
    "DOCUMENTO INSCRIPCION",
    "CAPITAL",
    "MATRICULA",
    "UBICACION",
}


@dataclass
class Appointment:
    section: str
    group: str
    cargo: str
    nombre: str
    identificacion_tipo: str
    identificacion_numero: str
    page: int


def strip_accents(value: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", value) if not unicodedata.combining(c)
    )


def norm(value: str) -> str:
    value = strip_accents(value).upper()
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def clean_space(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def extract_pages(pdf_path: Path) -> list[dict[str, Any]]:
    pages: list[dict[str, Any]] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for index, page in enumerate(pdf.pages, start=1):
            text = page.extract_text(x_tolerance=1, y_tolerance=3) or ""
            words = page.extract_words(x_tolerance=1, y_tolerance=3, keep_blank_chars=False)
            pages.append({"page": index, "text": text, "words": words})
    return pages


def extract_basic_info(full_text: str) -> dict[str, str]:
    patterns = {
        "razon_social": r"Raz[oó]n social:\s*(.+)",
        "nit": r"\bNit:\s*([0-9.\-\s]+)",
        "domicilio_principal": r"Domicilio principal:\s*(.+)",
        "direccion_domicilio_principal": r"Direcci[oó]n\s+del\s+domicilio\s+principal\s*:\s*(.*?)(?:\n|$)", #se cruza con linea 337
        "Teléfono comercial 1:": r"Tel[eé]fono comercial 1:\s*(.+)", # se cruza con linea 337
        "ciiu": r"C[oó]digo\s+CIIU\s*[:\-]?\s*(\d{4})"
    }
    result: dict[str, str] = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, full_text, flags=re.I)
        result[key] = clean_space(match.group(1)) if match else ""
    result["nit"] = re.sub(r"\s+", "-", result.get("nit", "")).strip("-")
    return result


def normalized_index(original: str) -> tuple[str, list[int]]:
    normalized_chars: list[str] = []
    index_map: list[int] = []
    for original_index, char in enumerate(original):
        decomposed = unicodedata.normalize("NFKD", char)
        for part in decomposed:
            if not unicodedata.combining(part):
                normalized_chars.append(part.upper())
                index_map.append(original_index)
    return "".join(normalized_chars), index_map


def section_between(full_text: str, start_terms: list[str], end_terms: list[str]) -> str:
    normalized, index_map = normalized_index(full_text)
    starts = [(normalized.find(term), term) for term in start_terms]
    starts = [(pos, term) for pos, term in starts if pos >= 0]
    if not starts:
        return ""
    start_pos, start_term = min(starts, key=lambda item: item[0])
    start_original = index_map[start_pos + len(start_term) - 1] + 1

    end_candidates: list[int] = []
    for term in end_terms:
        pos = normalized.find(term, start_pos + len(start_term))
        if pos >= 0:
            end_candidates.append(index_map[pos])
    end_original = min(end_candidates) if end_candidates else len(full_text)
    return clean_space(full_text[start_original:end_original])


def extract_faculties(full_text: str) -> dict[str, Any]:
    faculties_text = section_between(
        full_text,
        ["FACULTADES Y LIMITACIONES DEL REPRESENTANTE LEGAL", "FACULTADES Y LIMITACIONES"],
        ["NOMBRAMIENTOS", "REFORMAS DE ESTATUTOS", "RECURSOS", "CERTIFICA"],
    )
    limitations_area = norm(faculties_text if faculties_text else full_text)
    judicial_area = norm(full_text)

    limitation_patterns = [
        r"AUTORIZACION EXPRESA",
        r"CUANTIA",
        r"SUPERIOR A",
        r"NECESITAR[A-Z]* AUTORIZACION",
        r"LIMITACION",
        r"NO PODRA",
        r"JUNTA DE (MIEMBROS|SOCIOS|ACCIONISTAS|DIRECTIVA)",
    ]
    judicial_patterns = [
        r"REPRESENTANTES? JUDICIALES?",
        r"APODERAD[OA]S?",
        r"PODER GENERAL",
        r"ACTUACIONES JUDICIALES",
        r"AUDIENCIAS? DE CONCILIACION",
    ]

    limitations_found = sorted(
        {pattern for pattern in limitation_patterns if re.search(pattern, limitations_area)}
    )
    judicial_found = sorted(
        {pattern for pattern in judicial_patterns if re.search(pattern, judicial_area)}
    )
    return {
        "facultades_texto": faculties_text,
        "tiene_limitaciones": bool(limitations_found),
        "reglas_limitaciones_detectadas": limitations_found,
        "tiene_indicios_representacion_judicial": bool(judicial_found),
        "reglas_representacion_judicial_detectadas": judicial_found,
    }


def group_words_by_line(words: list[dict[str, Any]]) -> list[list[dict[str, Any]]]:
    lines: dict[int, list[dict[str, Any]]] = {}
    for word in words:
        top = round(float(word["top"]) / 3) * 3
        lines.setdefault(top, []).append(word)
    return [sorted(lines[top], key=lambda item: item["x0"]) for top in sorted(lines)]


def line_text(line: list[dict[str, Any]]) -> str:
    return clean_space(" ".join(str(word["text"]) for word in line))


def line_columns(line: list[dict[str, Any]]) -> tuple[str, str, str]:
    cargo_words: list[str] = []
    name_words: list[str] = []
    id_words: list[str] = []
    for word in line:
        x0 = float(word["x0"])
        text = str(word["text"])
        if x0 < 205:
            cargo_words.append(text)
        elif x0 < 370:
            name_words.append(text)
        else:
            id_words.append(text)
    return clean_space(" ".join(cargo_words)), clean_space(" ".join(name_words)), clean_space(" ".join(id_words))


def is_stop_line(raw: str) -> bool:
    normalized = norm(raw)
    if not normalized:
        return True
    if normalized.startswith("PAGINA "):
        return True
    if normalized.startswith(("POR ACTA", "POR DOCUMENTO", "POR ESCRITURA", "INSCRITA ")):
        return True
    return any(normalized.startswith(heading) for heading in STOP_HEADINGS)


def extract_appointments(pages: list[dict[str, Any]]) -> list[Appointment]:
    appointments: list[Appointment] = []
    active_table = False
    section = ""
    group = ""
    current: Appointment | None = None

    def flush_current() -> None:
        nonlocal current
        if current:
            current.cargo = clean_space(current.cargo)
            current.nombre = clean_space(current.nombre)
            appointments.append(current)
            current = None

    for page in pages:
        page_no = int(page["page"])
        for line in group_words_by_line(page["words"]):
            raw = line_text(line)
            normalized = norm(raw)

            if normalized in {"REPRESENTANTES LEGALES", "REPRESENTANTE LEGAL"}:
                flush_current()
                section = "REPRESENTANTES LEGALES"
                group = ""
                active_table = False
                continue
            if normalized in {"ORGANO DE ADMINISTRACION", "JUNTA DIRECTIVA"}:
                flush_current()
                section = "ORGANO DE ADMINISTRACION"
                active_table = False
                continue
            if normalized in {"REVISORES FISCALES", "PODERES"}:
                flush_current()
                section = ""
                group = ""
                active_table = False
                continue
            if normalized in {"PRINCIPALES", "SUPLENTES"}:
                flush_current()
                group = normalized
                active_table = False
                continue
            if "CARGO" in normalized and "NOMBRE" in normalized and "IDENTIFICACION" in normalized:
                flush_current()
                active_table = section in {"REPRESENTANTES LEGALES", "ORGANO DE ADMINISTRACION"}
                continue

            if not active_table:
                continue

            if is_stop_line(raw):
                flush_current()
                active_table = False
                continue

            cargo, name, ident = line_columns(line)
            id_match = DOC_ID_RE.search(ident)
            if id_match:
                flush_current()
                current = Appointment(
                    section=section or "NOMBRAMIENTOS",
                    group=group,
                    cargo=cargo,
                    nombre=name,
                    identificacion_tipo=clean_space(id_match.group(1).replace(" ", "")),
                    identificacion_numero=clean_space(id_match.group(2)),
                    page=page_no,
                )
                continue

            if current and not is_stop_line(raw):
                if cargo:
                    current.cargo = clean_space(current.cargo + " " + cargo)
                if name:
                    current.nombre = clean_space(current.nombre + " " + name)

        flush_current()
        active_table = False

    flush_current()
    return appointments


def extract_pdf(pdf_path: Path) -> dict[str, Any]:
    pages = extract_pages(pdf_path)
    full_text = "\n".join(page["text"] for page in pages)
    basic = extract_basic_info(full_text)
    appointments = extract_appointments(pages)
    return {
        "archivo": str(pdf_path),
        **basic,
        **extract_faculties(full_text),
        "nombramientos": [asdict(item) for item in appointments],
    }


def write_outputs(records: list[dict[str, Any]], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "camara_extraction.json").write_text(
        json.dumps(records, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    summary_rows = []
    appointment_rows = []
    for record in records:
        summary_rows.append(
            {
                "archivo": Path(record["archivo"]).name,
                "razon_social": record["razon_social"],
                "nit": record["nit"],
                "domicilio_principal": record["domicilio_principal"],
                "direccion_domicilio_principal": record["direccion_domicilio_principal"],  # nueva columna localizada en linea 102
                "Teléfono comercial 1:": record["Teléfono comercial 1:"],  # nueva columna localizada en linea 103
                "Actividad principal Código CIIU": record.get("ciiu", ""),
                "tiene_limitaciones": record["tiene_limitaciones"],
                "tiene_indicios_representacion_judicial": record[
                    "tiene_indicios_representacion_judicial"
                ],
            }
        )
        for item in record["nombramientos"]:
            appointment_rows.append({"archivo": Path(record["archivo"]).name, **item})

    excel_path = output_dir / "camara_extraction.xlsx"
    try:
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Resumen", index=False)
            pd.DataFrame(appointment_rows).to_excel(writer, sheet_name="Nombramientos", index=False)
    except PermissionError as exc:
        raise SystemExit(
            f"No pude escribir el Excel porque parece estar abierto o bloqueado: {excel_path}. "
            "Cierra el archivo y vuelve a correr el script."
        ) from exc


def find_pdfs(inputs: list[str]) -> list[Path]:
    pdfs: list[Path] = []
    for raw in inputs:
        path = Path(raw)
        if path.is_dir():
            pdfs.extend(sorted(path.glob("*.pdf")))
        elif path.suffix.lower() == ".pdf":
            pdfs.append(path)
    return pdfs


def run_extraction() -> None:
    """Lee los PDF de PDF_INPUT_DIR y escribe Excel + JSON en OUTPUT_DIR."""
    pdfs = find_pdfs([str(PDF_INPUT_DIR)])
    if not pdfs:
        raise SystemExit(
            f"No se encontraron PDFs en: {PDF_INPUT_DIR}\n"
            "Revisa PDF_INPUT_DIR en la celda de Configuración."
        )
    records = [extract_pdf(pdf) for pdf in pdfs]
    write_outputs(records, OUTPUT_DIR)
    for record in records:
        print(
            f"{Path(record['archivo']).name}: "
            f"{record['razon_social']} | NIT {record['nit']} | "
            f"{len(record['nombramientos'])} nombramientos"
        )
    print(f"\nSalida: {OUTPUT_DIR.resolve()}")


In [ ]:
# Ejecuta la extracción (PDFs de PDF_INPUT_DIR -> Excel + JSON en OUTPUT_DIR)
run_extraction()


# 2️⃣ Sección 2 — Pre-llenado de documentos

Genera 7 documentos por empresa a partir de las plantillas en `REQUIREMENTS_DIR`.

> **Importante:** ejecuta primero la celda **2.0** (carga el Excel y define los helpers compartidos). Las demás celdas usan ese `df` y esos helpers.

In [ ]:
# ============================================================================
# SECCIÓN 2.0 — Utilidades compartidas + carga del Excel  (EJECUTAR PRIMERO)
# ============================================================================
# Tercero NUEVO en esta sección:
#   lxml  - Manipulación del XML interno de los .docx  -> pip install lxml
# ============================================================================
from __future__ import annotations

from lxml import etree

W_NS   = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
XML_NS = "http://www.w3.org/XML/1998/namespace"


def fill_placeholders(xml_bytes: bytes, replacements: dict[str, str]) -> bytes:
    """
    Reemplaza placeholders del tipo [razon_social] y [nit_cam] en el XML.

    Primera pasada: reemplazo directo sobre el texto XML.
      → Funciona cuando el placeholder está íntegro en un solo <w:t>.

    Segunda pasada: opera SOLO dentro de la región separate→end de cada campo
      FORMTEXT, sin tocar los runs del label ni la estructura del campo.
      → Cubre el caso en que Word partió el placeholder en varios runs.
    """
    # ── Pasada 1: reemplazo directo ───────────────────────────────────────────
    xml_str = xml_bytes.decode("utf-8")
    for placeholder, value in replacements.items():
        xml_str = xml_str.replace(placeholder, value)
    xml_bytes = xml_str.encode("utf-8")

    # ── Pasada 2: runs partidos dentro de campos FORMTEXT ────────────────────
    root = etree.fromstring(xml_bytes)

    for sep_run in root.xpath(
        "//w:r[w:fldChar[@w:fldCharType='separate']]",
        namespaces={"w": W_NS},
    ):
        parent = sep_run.getparent()
        siblings = list(parent)
        sep_idx = siblings.index(sep_run)

        # Recorre siblings siguientes hasta fldChar end
        value_runs: list[tuple] = []
        for elem in siblings[sep_idx + 1 :]:
            if elem.tag != f"{{{W_NS}}}r":
                continue
            if elem.xpath("w:fldChar[@w:fldCharType='end']", namespaces={"w": W_NS}):
                break
            t = elem.find(f"{{{W_NS}}}t")
            if t is not None:
                value_runs.append((elem, t))

        if not value_runs:
            continue

        full_text = "".join(t.text or "" for _, t in value_runs)
        changed = False
        for placeholder, value in replacements.items():
            if placeholder in full_text:
                full_text = full_text.replace(placeholder, value)
                changed = True

        if changed:
            _, first_t = value_runs[0]
            first_t.text = full_text
            first_t.set(f"{{{XML_NS}}}space", "preserve")
            for _, t in value_runs[1:]:
                t.text = ""

    return etree.tostring(root, xml_declaration=True, encoding="UTF-8", standalone=True)


def create_filled_docx(
    template_path: Path,
    razon_social: str,
    nit: str,
    output_path: Path,
) -> None:
    """Copia la plantilla y reemplaza [razon_social] y [nit]."""
    replacements = {
        "[razon_social]": razon_social,
        "[nit_cam]": nit,
    }
    with zipfile.ZipFile(template_path, "r") as zin:
        with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                data = zin.read(item.filename)
                if item.filename == "word/document.xml":
                    data = fill_placeholders(data, replacements)
                zout.writestr(item, data)


def safe_filename(name: str) -> str:
    """Elimina caracteres no permitidos en nombres de archivo."""
    return re.sub(r'[\\/:*?"<>|]', "_", name).strip()


# ── Carga única del Excel: todas las formas de la Sección 2 usan este df ──────
df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")
print(f"{len(df)} empresa(s) cargada(s) desde: {EXCEL_PATH}")


## 2.1 — Annex A - Signature Card for Accounts in COP

In [ ]:
# ── 2.1  Annex A - Signature Card for Accounts in COP ────────────────────────
processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - Annex A.docx"
    output_path = OUTPUT_DIR / filename

    create_filled_docx(ANNEX_A_TEMPLATE, razon_social, nit, output_path)
    print(f"  OK  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


## 2.2 — Contact Information Certificate For Electronic Signatures

In [ ]:
processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print(f"  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - Contact Information Certificate.docx"
    output_path = OUTPUT_DIR / filename

    create_filled_docx(CONTACT_TEMPLATE, razon_social, nit, output_path)
    print(f"  ✓  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


## 2.3 — CRS FATCA Form

In [ ]:
from pypdf import PdfReader, PdfWriter

BOGOTA = "Bogotá D.C."

def build_crs_fields(row: dict) -> tuple[dict, dict]:
    """Devuelve (campos_pag4, campos_pag11) a partir de una fila del Excel."""
    domicilio = str(row.get("domicilio_principal", "")).strip()
    is_bogota = domicilio == BOGOTA

    p4 = {
        "Text16": str(row.get("razon_social", "")).strip(),
        "Text17": "Colombia" if is_bogota else "",
        "Text18": str(row.get("direccion_domicilio_principal", "")).strip(),
        "Text19": "Bogota"   if is_bogota else "",
        "Text20": "Bogota"   if is_bogota else "",
        "Text21": "Colombia" if is_bogota else "",
        "Text22": "111111"   if is_bogota else "",
    }
    p11 = {
        "CJTaxResidence1": "Colombia" if is_bogota else "",
        "TIN1":            str(row.get("nit", "")).strip(),
    }
    return p4, p11

def fill_crs_pdf(template_path: Path, p4: dict, p11: dict, output_path: Path) -> None:
    """Rellena los campos del CRS FATCA Form y guarda en output_path."""
    reader = PdfReader(str(template_path))
    writer = PdfWriter()
    writer.append(reader)
    writer.update_page_form_field_values(writer.pages[3],  p4)
    writer.update_page_form_field_values(writer.pages[10], p11)
    with open(output_path, "wb") as f:
        writer.write(f)

# ── Ejecución ────────────────────────────────────────────────────────────────

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    p4, p11 = build_crs_fields(row)
    filename    = f"{safe_filename(razon_social)} - CRS FATCA Form.pdf"
    output_path = OUTPUT_DIR / filename

    fill_crs_pdf(CRS_TEMPLATE, p4, p11, output_path)
    print(f"  ✓  {filename}  |  NIT: {p11['TIN1']}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


## 2.4 — Electronic Signatures Services Terms

In [ ]:
processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - Electronic Signatures Services Terms.docx"
    output_path = OUTPUT_DIR / filename

    create_filled_docx(ESST_TEMPLATE, razon_social, nit, output_path)
    print(f"  ✓  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


## 2.5 — Formulario Grandes Exposiciones

In [ ]:
import shutil
from datetime import date
from openpyxl import load_workbook

def parse_nit(nit_raw: str) -> tuple[str, str]:
    """
    Separa el NIT en (número, dígito_verificación).
    Ejemplo: '860034594-1' → ('860034594', '1')
    Si no hay guion devuelve (nit_raw, '').
    """
    if "-" in nit_raw:
        parts = nit_raw.split("-", 1)
        return parts[0].strip(), parts[1].strip()
    return nit_raw.strip(), ""

def fill_grandes_exposiciones(template_path: Path, row: dict, output_path: Path) -> None:
    """Copia el template y rellena las celdas C5–C11."""
    shutil.copy2(str(template_path), str(output_path))
    wb = load_workbook(str(output_path))
    ws = wb.active

    razon_social = str(row.get("razon_social", "")).strip()
    nit_raw      = str(row.get("nit", "")).strip()
    nit_num, nit_dig = parse_nit(nit_raw)

    # C5 — Fecha de diligenciamiento (DD/MM/YYYY)
    ws["C5"] = date.today()
    ws["C5"].number_format = "DD/MM/YYYY"

    # C6 — Nombre / Razón Social
    ws["C6"] = razon_social

    # C7 — Tipo ID (dropdown: NIT | TIN | OTHER)
    ws["C7"] = "NIT" if nit_raw else None

    # C8 — Dígito de verificación (dropdown: 1-9 | not applicable.)
    if nit_dig:
        try:
            ws["C8"] = int(nit_dig)
        except ValueError:
            ws["C8"] = nit_dig

    # C9 — Número de identificación (dígitos antes del guion)
    if nit_num:
        try:
            ws["C9"] = int(nit_num)
        except ValueError:
            ws["C9"] = nit_num

    # C10 — Nombre Representante Legal (placeholder)
    ws["C10"] = "[por favor diligenciar]"

    # C11 — Identificación Representante Legal (placeholder)
    ws["C11"] = "[por favor diligenciar]"

    wb.save(str(output_path))

# ── Ejecución ────────────────────────────────────────────────────────────────

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - Formulario Grandes Exposiciones.xlsx"
    output_path = OUTPUT_DIR / filename

    fill_grandes_exposiciones(FORM_GE_TEMPLATE, row, output_path)
    print(f"  ✓  {filename}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


## 2.6 — GCC - Declaración de Veracidad

In [ ]:
from pypdf import PdfReader, PdfWriter

def fill_gcc_pdf(template_path: Path, razon_social: str, nit: str, output_path: Path) -> None:
    """Rellena los campos AcroForm del GCC-Declaracion de Veracidad."""
    reader = PdfReader(str(template_path))
    writer = PdfWriter()
    writer.append(reader)
    writer.update_page_form_field_values(writer.pages[0], {
        "Nombre de la sociedad/empresa": razon_social,
        "(NIT)":                         nit,
    })
    with open(output_path, "wb") as f:
        writer.write(f)

# ── Ejecución ────────────────────────────────────────────────────────────────

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - GCC-Declaracion de Veracidad.pdf"
    output_path = OUTPUT_DIR / filename

    fill_gcc_pdf(GCC_TEMPLATE, razon_social, nit, output_path)
    print(f"  ✓  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


## 2.7 — KYC application form (Onboarding) v. Mar 2026

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
from pypdf import PdfReader, PdfWriter

BOGOTA = "Bogotá D.C."

def build_kyc_fields(row: dict) -> dict:
    """
    Construye el diccionario de campos AcroForm a partir de una fila del Excel.

    Mapeo de campos:
      Fecha Date               ← Hoy  (DD/MM/YYYY)
      Text2  [nit_cam]         ← nit             (col C)
      Text3  [razon_social]    ← razon_social     (col B)
      Text4  [Teléfono…]       ← Teléfono comercial 1: (col F)
      Text5  [address]         ← direccion_domicilio_principal (col E)
      Text6  [pais]            ← "Colombia" si domicilio == "Bogotá D.C.", si no ""
      Código CIIU ISIC Code    ← Actividad principal Código CIIU (col G)
    """
    domicilio    = str(row.get("domicilio_principal", "")).strip()
    is_bogota    = domicilio == BOGOTA
    telefono_raw = row.get("Teléfono comercial 1:", "")
    ciiu_raw     = row.get("Actividad principal Código CIIU", "")

    return {
        "Fecha Date":            date.today().strftime("%d/%m/%Y"),
        "Text2":                 str(row.get("nit", "")).strip(),
        "Text3":                 str(row.get("razon_social", "")).strip(),
        "Text4":                 str(telefono_raw).strip() if pd.notna(telefono_raw) else "",
        "Text5":                 str(row.get("direccion_domicilio_principal", "")).strip(),
        "Text6":                 "Colombia" if is_bogota else "",
        "Código CIIU ISIC Code": str(ciiu_raw).strip() if pd.notna(ciiu_raw) else "",
    }

def fill_kyc_pdf(template_path: Path, fields: dict, output_path: Path) -> None:
    """Rellena los campos AcroForm del KYC form en todas las páginas del PDF."""
    reader = PdfReader(str(template_path))
    writer = PdfWriter()
    writer.append(reader)
    for page in writer.pages:
        writer.update_page_form_field_values(page, fields)
    with open(output_path, "wb") as f:
        writer.write(f)

# ── Ejecución ────────────────────────────────────────────────────────────────

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    fields      = build_kyc_fields(row)
    filename    = f"{safe_filename(razon_social)} - KYC application form.pdf"
    output_path = OUTPUT_DIR / filename

    fill_kyc_pdf(KYC_TEMPLATE, fields, output_path)
    print(f"  ✓  {filename}  |  NIT: {fields['Text2']}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {OUTPUT_DIR.resolve()}")


# 3️⃣ Sección 3 — Folder Creation

Por cada empresa crea en la red `[razón social] - [nit]` con subcarpetas **Only MO** y **Only Maker**, y copia todos los documentos de `OUTPUT_DIR` dentro de **Only MO**. Si el folder ya existe, **aborta con error** (no sobreescribe). Usa el mismo `df` cargado en 2.0.

In [ ]:
# ============================================================================
# SECCIÓN 3 — Folder Creation (carpetas de contraparte en la red)
# ============================================================================
# Usa el df cargado en 2.0 y NETWORK_BASE_PATH / OUTPUT_DIR de Configuración.
# ============================================================================


def _safe_folder_name(name: str) -> str:
    """Elimina caracteres no permitidos en nombres de carpeta en Windows."""
    return re.sub(r'[/:*?"<>|]', "_", name).strip()


def create_counterparty_folders() -> None:
    for _, row in df.iterrows():
        razon_social = str(row.get("razon_social", "")).strip()
        nit          = str(row.get("nit", "")).strip()

        if not razon_social:
            print(f"  [SKIP] Fila sin razón social: {row.to_dict()}")
            continue

        folder_name      = _safe_folder_name(f"{razon_social} - {nit}")
        counterparty_dir = NETWORK_BASE_PATH / folder_name

        if counterparty_dir.exists():
            raise FileExistsError(
                f"El folder ya existe y no se sobreescribirá: {counterparty_dir}\n"
                f"Elimínalo manualmente si deseas regenerarlo."
            )

        only_mo_dir    = counterparty_dir / "Only MO"
        only_maker_dir = counterparty_dir / "Only Maker"
        only_mo_dir.mkdir(parents=True)
        only_maker_dir.mkdir(parents=True)

        files_copied = 0
        for file in OUTPUT_DIR.iterdir():
            if file.is_file():
                shutil.copy2(file, only_mo_dir / file.name)
                files_copied += 1

        print(f"  Folder creado  : {folder_name}")
        print(f"     Only MO    : {files_copied} archivo(s) copiado(s)")
        print(f"     Only Maker : (vacío)")


create_counterparty_folders()
